**Human Movement Intelligence | Multimodal Movement Learning | Presented by: Reza Saadatyar (2026)**

**02 | Signal Processing, Aligned Windows and Participant-Disjoint Splits**

![Paired E1 participant cohort, label-pure windowing, statistical sensor representations and participant-disjoint evaluation](../../images/02_signal_processing_and_aligned_windows.svg)

**Research question:** How can NinaPro DB2 exercise-one (E1) recordings be transformed into paired, label-consistent sEMG and acceleration windows while keeping people independent across model development and final testing?

**Source:** The original E1 recordings from **40 participants (S01–S40)**, using the corrected class definitions **0–17** established in [Notebook 01](01_ninapro_pairing_and_label_audit.ipynb).

**Observed cohort:** **337,853 eligible source windows** were identified. A predetermined limit of **1,000 windows per person** retained **40,000 paired windows** of **300 ms** each, with a **100-ms step** on the common released sample grid.

**Representations:** Each selected window provides **48 sEMG** and **96 acceleration** fixed statistical features; the paired **600-row temporal signals** remain available for later neural models.

**Evaluation design:** **35 Development participants** in five person-disjoint folds and **five reserved Test participants**. This notebook constructs the dataset and evaluation assignments; it does not train or select movement classifiers.


**00. Colab Repository Setup**

In [1]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


/content/human-movement-intelligence


**01. Imports and Research Environment**

In [2]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("multimodal_movement_learning")

# ----------------- 2. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 3. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

plot_style(
    font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(9, 4),
)

# ------------------------ 4. Study Research Utilities ------------------------
from ninapro_source import ensure_subject_archive, load_exercise, inspect_record
from ninapro_dataset import (
    build_window_registry, select_windows_uniformly, save_source_arrays,
    construct_feature_arrays, assign_participants, check_artifact_alignment,
)

# ---------------------- 5. Source and Output Locations -----------------------
study_dir = root / "multimodal_movement_learning"
output_dir = study_dir / "output"
processed_dir = root / "data" / "processed" / "ninapro_db2"
raw_dir = root / "data" / "raw" / "ninapro_db2"
output_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
print("Research output directory:", output_dir)


Local changes kept; continuing with the current checkout. Conflicting Git update deferred.
Research output directory: /content/human-movement-intelligence/multimodal_movement_learning/output


**02. Load the Source Label Definitions**

Read the source audit and corrected E1 label dictionary established in Notebook 01. The target codes comprise **Rest (0)** and **seventeen finger-movement classes (1–17)**. This shared class order is retained across subsequent unimodal and multimodal comparisons.

The source label definitions are checked against the recordings of the wider cohort. Class labels and repetition identifiers are used to define target values and valid recording intervals, not as predictor inputs.


In [3]:
# ================ 02. Load the Verified Source Label Contract ================

NOTEBOOK_NUMBER = 2
source_labels = pd.read_csv(output_dir / "01_label_dictionary.csv")
source_audit = pd.read_csv(output_dir / "01_pairing_registry.csv")
classes = np.sort(source_labels.label.to_numpy(int))
if len(classes) < 2 or not np.array_equal(classes, np.unique(classes)):
    raise ValueError("The audited E1 label dictionary has fewer than two valid classes.")
if int(source_audit.loc[0, "exercise_id"]) != 1:
    raise ValueError("Notebook 01 must audit the source exercise E1.")
print("Source-derived target classes:", classes.tolist())


Source-derived target classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]


**03. Inspect and Segment All E1 Recordings**

Process **40 participant recordings** in subject order. For each E1 source, verify its corrected labels and paired sEMG–acceleration rows, then identify candidate windows within contiguous segments of the same corrected label and repetition.

A window spans **300 ms (600 released rows)** and advances by **100 ms (200 rows)**. Windows crossing an annotation boundary or containing a non-finite sample in either modality are excluded. This produced **337,853 eligible source windows** across the 40 participants. The source recordings are handled separately for each person, so the entire cohort need not be loaded into memory at once.


In [4]:
# ============= 03. Ingest the First Exercise Across Participants =============

# --------------------- 1. Subject and Window Parameters ----------------------
SUBJECT_NUMBERS = tuple(range(1, 41))
DOWNLOAD_MISSING = True
WINDOW_MS = 300
STEP_MS = 100
MAX_WINDOWS_PER_SUBJECT = 1000  # Predeclared, time-uniform, label-blind compute budget.
TEST_PEOPLE = 5
DEVELOPMENT_FOLDS = 5
SPLIT_SEED = 42

# ------------------- 2. Ingest and Window the Source Data --------------------
source_rows, window_parts, trial_parts = [], [], []
for number in SUBJECT_NUMBERS:
    archive = ensure_subject_archive(raw_dir, number, download=DOWNLOAD_MISSING)
    record = load_exercise(archive, number, exercise=1)
    actual = set(np.unique(record.restimulus).tolist())
    if not actual.issubset(set(classes)):
        raise ValueError(f"Unexpected E1 corrected labels in {record.subject_id}: {actual-set(classes)}")
    window_table, trial_table = build_window_registry(
        record, set(classes), window_ms=WINDOW_MS, step_ms=STEP_MS,
    )
    if window_table.empty:
        raise ValueError(f"No valid paired E1 windows in {record.subject_id}.")
    save_source_arrays(record, processed_dir)
    source_rows.append(inspect_record(record))
    window_parts.append(window_table)
    trial_parts.append(trial_table)
    print(f"{record.subject_id}: {len(window_table):,} eligible source windows")

# -------------------- 3. Consolidate the Source Inventory --------------------
source_inventory = pd.DataFrame(source_rows)
all_windows = pd.concat(window_parts, ignore_index=True)
all_trials = pd.concat(trial_parts, ignore_index=True)
print("Eligible source participants:", source_inventory.subject_id.nunique())
print("Full source-window inventory:", len(all_windows))


S01: 8,532 eligible source windows
S02: 8,509 eligible source windows
S03: 8,466 eligible source windows
S04: 8,493 eligible source windows
S05: 8,497 eligible source windows
S06: 8,478 eligible source windows
S07: 8,478 eligible source windows
S08: 8,511 eligible source windows
S09: 8,497 eligible source windows
S10: 8,523 eligible source windows
S11: 8,514 eligible source windows
S12: 8,476 eligible source windows
S13: 8,503 eligible source windows
S14: 8,513 eligible source windows
S15: 8,519 eligible source windows
S16: 8,506 eligible source windows
S17: 8,523 eligible source windows
S18: 8,497 eligible source windows
S19: 8,481 eligible source windows
S20: 8,499 eligible source windows
S21: 8,559 eligible source windows
S22: 8,542 eligible source windows
S23: 8,505 eligible source windows
S24: 8,491 eligible source windows
S25: 8,517 eligible source windows
S26: 8,280 eligible source windows
S27: 8,374 eligible source windows
S28: 8,388 eligible source windows
S29: 8,345 eligible 

**04. Apply the Fixed Participant-Level Window Budget**

Select up to **1,000 windows for each participant** from their eligible windows, using evenly spaced indices in **eligible-window order**. This deterministic selection depends neither on class labels nor on model performance. In the executed cohort, all 40 people reached the cap, producing **40,000 paired windows**.

This is a **reduced analytical subset**, not the full eligible E1 window inventory. The procedure is neither random sampling nor class balancing: label frequencies after selection may differ from those in the complete source recordings. The retained `window_key` preserves the source interval, while `window_row` defines the common numerical-array ordering for both modalities.


In [5]:
# ================== 04. Select Time-Uniform Paired Windows ===================

index = select_windows_uniformly(all_windows, MAX_WINDOWS_PER_SUBJECT)
observed = (index.groupby(["subject_id", "label"]).size()
            .unstack(fill_value=0).reindex(columns=classes, fill_value=0))
if (observed.sum(axis=0) == 0).any():
    raise ValueError("The predeclared window selection omitted an audited E1 class.")
print("Retained paired windows:", len(index))
display(observed.head())
display(index.head())


Retained paired windows: 40000


label,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
subject_id,,,,,,,,,,,,,,,,,,
S01,592,45,32,28,24,22,12,33,14,23,25,24,25,23,20,11,24,23
S02,500,33,38,28,33,37,23,44,31,32,30,30,26,22,25,20,25,23
S03,519,45,44,40,23,21,36,43,39,31,15,33,19,19,15,21,19,18
S04,353,44,45,44,42,45,29,44,42,42,42,44,32,21,22,45,38,26
S05,607,46,28,32,19,12,24,39,16,23,12,22,16,15,13,44,20,12


,window_row,window_key,subject_id,exercise_id,segment_id,repetition_id,label,start_sample,end_sample,start_s,end_s,source_file
0,0,S01_E1_G00001_000000000,S01,1,S01_E1_G00001,1,0,0,600,0.0,0.3,DB2_s1/S1_E1_A1.mat
1,1,S01_E1_G00001_000001600,S01,1,S01_E1_G00001,1,0,1600,2200,0.8,1.1,DB2_s1/S1_E1_A1.mat
2,2,S01_E1_G00001_000003400,S01,1,S01_E1_G00001,1,0,3400,4000,1.7,2.0,DB2_s1/S1_E1_A1.mat
3,3,S01_E1_G00001_000005000,S01,1,S01_E1_G00001,1,0,5000,5600,2.5,2.8,DB2_s1/S1_E1_A1.mat
4,4,S01_E1_G00001_000006800,S01,1,S01_E1_G00001,1,0,6800,7400,3.4,3.7,DB2_s1/S1_E1_A1.mat


**05. Construct Paired Statistical and Temporal Representations**

Extract four sEMG statistics per channel: **root mean square**, **mean absolute value**, **normalised waveform length**, and **log variance**. The **12 channels × 4 features** yield an sEMG feature matrix of **40,000 × 48**.

For acceleration, compute the **mean and standard deviation of each of 36 axes**, plus the **mean and standard deviation of the magnitude for each of 12 three-axis sensor groups**. These 72 + 24 quantities form **40,000 × 96** acceleration features.

Both representations use precisely the same **40,000 source-window keys**. Temporal models subsequently read the corresponding **600 × 12 sEMG** and **600 × 36 acceleration** samples from the released common-grid recordings. The accelerometer rows are represented on that grid; they are **not independent native 2,000-Hz acceleration measurements**. Fixed feature extraction requires no cohort-fitted scaling; any learned normalisation must be estimated from the permitted Training participants in the modelling stage.


In [6]:
# ======== 05. Construct Statistical Features on the Same Window Keys =========

emg_features, acc_features = construct_feature_arrays(index, processed_dir)
check_artifact_alignment(index, (emg_features, acc_features))
np.save(processed_dir / "02_emg_features.npy", emg_features)
np.save(processed_dir / "02_acc_features.npy", acc_features)
print("Statistical sEMG features:", emg_features.shape)
print("Statistical Acc features:", acc_features.shape)
print("Neural source-window samples:",
      int(index.end_sample.iloc[0] - index.start_sample.iloc[0]))
print("Neural channels: sEMG 12 / acceleration 36")


Statistical sEMG features: (40000, 48)
Statistical Acc features: (40000, 96)
Neural source-window samples: 600
Neural channels: sEMG 12 / acceleration 36


**06. Establish Participant-Disjoint Evaluation Assignments**

Split the cohort at **participant level**, not at window level. The deterministic partition (`SPLIT_SEED = 42`) assigns **35 people to Development** and reserves **S05, S08, S19, S33 and S40** for final Test evaluation.

The Development group is divided into **five validation folds**, each containing **seven distinct participants**. No Test participant appears in a Development fold. Overlapping windows from the same source recording always remain with their participant, avoiding direct cross-person split leakage. Later modelling stages fit scalers and select methods using Development data only.


In [7]:
# ======== 06. Freeze Participant-Disjoint Development and Test Roles =========

participant_roles, folds = assign_participants(
    index, test_people=TEST_PEOPLE, folds=DEVELOPMENT_FOLDS,
    seed=SPLIT_SEED,
)
assert not set(participant_roles.loc[participant_roles.role == "Test", "subject_id"]) & set(folds.subject_id)
assert set(index.subject_id) == set(participant_roles.subject_id)
display(participant_roles.groupby("role", as_index=False).agg(people=("subject_id", "nunique")))
display(folds.groupby("validation_fold", as_index=False).agg(people=("subject_id", "nunique")))


,role,people
0,Development,35
1,Test,5


,validation_fold,people
0,0,7
1,1,7
2,2,7
3,3,7
4,4,7


**07. Examine Selected Class Support and Signal Completeness**

The left panel reports **window counts for all eighteen corrected E1 classes** after applying the fixed per-person selection. These are counts of potentially overlapping windows, not independent movement events or independent participants.

The right panel summarises finite-value coverage from the original **40 participant E1 recordings**. The observed mean finite fraction is **1.00 for both sEMG and acceleration**. Full finite coverage does not establish artefact-free recording, native accelerometer bandwidth, correct movement annotations or independent sensor-clock precision. Neither panel reports classifier performance.


In [ ]:
# =========== 07. Inspect Cohort Balance and Paired Channel Quality ===========

quality = pd.DataFrame({
    "modality": ["sEMG", "Acceleration"],
    "mean_finite_fraction": [source_inventory.emg_finite_fraction.mean(),
                             source_inventory.acc_finite_fraction.mean()],
})
display(quality)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
axes[0].bar([str(v) for v in classes], observed.sum(axis=0).to_numpy())
axes[0].set(title="Selected E1 class support", xlabel="Corrected E1 label", ylabel="Paired windows")
axes[1].bar(quality.modality, quality.mean_finite_fraction)
axes[1].set(title="Finite-value coverage", ylabel="Mean finite fraction", ylim=(0, 1.05))
for ax in axes:
    ax.grid(axis="y", alpha=0.2)
save_figure(fig, output_dir / "02_channel_quality.png")
plt.show()


**08. Preserve Source, Window and Split Provenance**

Maintain a traceable record of the source cohort, valid label/repetition segments and the selected paired windows. Each selected observation retains its **participant ID, corrected label, repetition, source interval, unique `window_key` and integer `window_row`**.

The independently assigned Development/Test roles and Development validation folds provide the fixed evaluation contract for subsequent experiments. The numerical feature arrays and original-length source signals remain separate from these small metadata tables; their row and subject identifiers must agree when the next notebook loads them.


In [9]:
# ============= 08. Save Trial, Window and Person-Fold Contracts ==============

all_trials.to_csv(output_dir / "02_aligned_trial_index.csv", index=False)
index.to_csv(output_dir / "02_window_index.csv", index=False)
participant_roles.to_csv(output_dir / "02_participant_split_assignments.csv", index=False)
folds.to_csv(output_dir / "02_development_fold_assignments.csv", index=False)
source_inventory.to_csv(output_dir / "02_source_inventory.csv", index=False)
print("Final paired window rows:", len(index))
print("Independent people:", index.subject_id.nunique())


Final paired window rows: 40000
Independent people: 40


**09. Research Summary and Output Download**

The completed E1 preparation identifies **337,853 eligible windows** from **40 participants** and retains **40,000 paired windows** using a fixed participant-level budget. The feature matrices contain **48 sEMG** and **96 acceleration** values per window; person-level separation reserves five people for final Test evaluation.

Notebook 03 uses the paired-window index, participant assignments and class definitions together with the corresponding numerical signal and feature arrays. These are **data-preparation findings**, not movement-recognition performance estimates.

The final cell packages the **six current research outputs** and lists their filenames once alongside the Colab download location.


In [11]:
# ================= 09. Research Summary and Output Download ==================

download_outputs_from_colab(
    folder="multimodal_movement_learning/output",
    stage=f"{NOTEBOOK_NUMBER:02d}",
)


------------------------------------------------------------------------
Notebook 02
Files packaged: 6
  + 02_aligned_trial_index.csv
  + 02_channel_quality.png
  + 02_development_fold_assignments.csv
  + 02_participant_split_assignments.csv
  + 02_source_inventory.csv
  + 02_window_index.csv
Download ZIP from Colab Contents: /content/multimodal_movement_learning_Notebook_02_Outputs.zip
